# Condo fees in Aclimação — apartments for sale

How does the monthly condo fee (`condo_fee`) of an apartment listed **for sale** in
Aclimação compare to a **R$ 2.000/month** budget?

Every chart is a histogram of `condo_fee` against that R$ 2.000 reference, cut by a
different group of apartments (bedrooms, usable area, parking, asking price), so the
question "which kind of apartment actually fits the budget?" has a per-group answer.

**Layout of the notebook**

1. Setup and chart theme
2. Load the listings from `fact_listings`
3. Remove garbage data (audited, nothing dropped silently)
4. Feature engineering — the groups being compared
5. Table view — every number the charts show
6. The histograms
7. Takeaways

## 1. Setup and chart theme

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from dotenv import load_dotenv

# The notebook lives in notebooks/, the package lives in the project root.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.database import db_manager  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
print(f"project root: {PROJECT_ROOT}")

### Theme

Colours come from a validated categorical palette: only the first three slots are used
(blue → orange → aqua), assigned **in fixed order** so a group keeps its colour across
every chart. Both light and dark steps are defined — flip `DARK_MODE` and re-run the
plotting cells.

In [ ]:
DARK_MODE = False

THEMES = {
    "light": dict(
        template="plotly_white",
        series=["#2a78d6", "#eb6834", "#1baf7a"],  # blue, orange, aqua
        surface="#fcfcfb",
        ink="#0b0b0b",
        ink_2="#52514e",
        muted="#898781",
        grid="#e1e0d9",
        axis="#c3c2b7",
    ),
    "dark": dict(
        template="plotly_dark",
        series=["#3987e5", "#d95926", "#199e70"],  # same hues, stepped for dark
        surface="#1a1a19",
        ink="#ffffff",
        ink_2="#c3c2b7",
        muted="#898781",
        grid="#2c2c2a",
        axis="#383835",
    ),
}
T = THEMES["dark" if DARK_MODE else "light"]
pio.templates.default = T["template"]

# The budget every distribution is compared against
BUDGET = 2_000


def brl(value: float, decimals: int = 0) -> str:
    """Format a number the Brazilian way: R$ 1.234,56"""
    text = f"{value:,.{decimals}f}"
    return "R$ " + text.replace(",", "~").replace(".", ",").replace("~", ".")


print(f"theme: {'dark' if DARK_MODE else 'light'} | budget: {brl(BUDGET)}")

## 2. Load the listings

In [ ]:
QUERY = """
    SELECT
        listing_id,
        condo_fee,
        price,
        total_area_m2,
        bedrooms,
        bathrooms,
        vacancies,
        floor,
        construction_year,
        street_address,
        location_type,
        advertizer,
        url,
        listing_date,
        updated_at
    FROM fact_listings
    WHERE neighborhood in %(neighborhood)s
      AND business_type = %(business_type)s
      AND unit_type = %(unit_type)s
"""
FILTERS = {
    "neighborhood": ("Aclimação","Vila Mariana"),   # the DB stores it accented: "Aclimação"
    "business_type": "SALE",
    "unit_type": "APARTMENT",
}

with db_manager.get_connection() as conn:
    raw = pd.read_sql(QUERY, con=conn, params=FILTERS)

print(f"{len(raw):,} raw apartment-for-sale listings in Aclimação")
raw.head()

## 3. Remove garbage data

`condo_fee` is scraped from a free-text field on the listing, so it carries three kinds
of junk: **not disclosed** (stored as `0`), **the same flat listed twice**, and
**wrong-field entries** — the worst one in this dataset is a "condo fee" of
R$ 950.000, which is the asking price pasted into the fee box.

Rather than hand-picking thresholds, the implausible-fee rule is a Tukey fence on
**fee per m²**: a fee only means something relative to the size of the flat, and the
fence adapts to the neighbourhood instead of to a guess. Every step is counted below so
nothing disappears quietly.

In [ ]:
audit = []


def drop(df: pd.DataFrame, mask: pd.Series, reason: str) -> pd.DataFrame:
    """Drop rows where mask is True and record why."""
    audit.append({"step": reason, "listings_removed": int(mask.sum())})
    return df.loc[~mask].copy()


clean = raw.copy()
audit.append({"step": "starting listings", "listings_removed": 0})

# 1. the same flat scraped twice under the same URL
clean = drop(clean, clean["url"].duplicated(), "duplicate listing (same URL)")

# 2. fee not disclosed -- the scraper stores a missing monthlyCondoFee as 0
clean = drop(clean, clean["condo_fee"].isna() | (clean["condo_fee"] <= 0),
             "condo_fee missing or zero")

# 3. no area / no price -- can't be normalised or sanity-checked
clean = drop(clean, clean["total_area_m2"] <= 0, "total_area_m2 is zero")
clean = drop(clean, clean["price"] <= 0, "price is zero")

# 4. implausible fee for the size of the flat
clean["fee_per_m2"] = clean["condo_fee"] / clean["total_area_m2"]
q1, q3 = clean["fee_per_m2"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
LOWER_FLOOR = 1.0  # below R$1/m2 a month is not a real condo fee
clean = drop(
    clean,
    (clean["fee_per_m2"] < LOWER_FLOOR) | (clean["fee_per_m2"] > upper_fence),
    f"fee per m2 outside R$ {LOWER_FLOOR:.0f}–{upper_fence:.0f}/m2 (Tukey fence)",
)

audit_table = pd.DataFrame(audit)
audit_table.loc[audit_table["step"] == "starting listings", "listings_removed"] = len(raw)
audit_table["remaining"] = len(raw) - audit_table["listings_removed"].cumsum() + len(raw)
audit_table.loc[0, "remaining"] = len(raw)
audit_table.loc[1:, "remaining"] = (
    len(raw) - audit_table.loc[1:, "listings_removed"].cumsum()
)
audit_table

In [ ]:
removed = len(raw) - len(clean)
print(f"kept {len(clean):,} of {len(raw):,} listings "
      f"({removed:,} removed, {removed / len(raw):.1%} garbage)")
print()
print(clean[["condo_fee", "fee_per_m2", "price", "total_area_m2"]]
      .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1))

## 4. The groups being compared

Each grouping is an ordered set of buckets, and each bucket keeps a stable label so the
same group reads the same way in every chart and table.

In [ ]:
BEDROOM_ORDER = ["≤ 1 bedroom", "2 bedrooms", "3 bedrooms", "4+ bedrooms"]
AREA_ORDER = ["< 60 m²", "60–99 m²", "100–160 m²", "≥ 160 m²"]
PARKING_ORDER = ["No parking", "1 space", "2+ spaces"]
PRICE_ORDER = ["≤ R$ 700 mil", "R$ 700 mil – 1,2 mi", "> R$ 1,2 mi"]

df = clean.copy()

df["bedroom_group"] = pd.cut(
    df["bedrooms"], bins=[-1, 1, 2, 3, 99], labels=BEDROOM_ORDER, ordered=True
)
df["area_group"] = pd.cut(
    df["total_area_m2"], bins=[0, 60, 100, 160, 10_000], labels=AREA_ORDER,
    right=False, ordered=True,
)
df["parking_group"] = pd.cut(
    df["vacancies"], bins=[-1, 0, 1, 99], labels=PARKING_ORDER, ordered=True
)
df["price_group"] = pd.cut(
    df["price"], bins=[0, 700_000, 1_200_000, 10**9], labels=PRICE_ORDER,
    right=True, ordered=True,
)

# The two-way cuts the question is really about
df["bedrooms_2plus"] = np.where(df["bedrooms"] >= 2, "2+ bedrooms", "1 bedroom or studio")
df["area_160plus"] = np.where(df["total_area_m2"] >= 160, "≥ 160 m²", "< 160 m²")
df["family_size"] = np.where(
    (df["bedrooms"] >= 3) & (df["total_area_m2"] <= 160),
    "3+ bedrooms and <= 160 m²",
    "Everything else",
)
df["within_budget"] = df["condo_fee"] <= BUDGET

df[["condo_fee", "bedroom_group", "area_group", "parking_group",
    "price_group", "within_budget"]].head()

## 5. Table view

Every value the histograms encode, in text — so nothing below depends on reading a
colour or hovering a bar.

In [ ]:
def summarise(frame: pd.DataFrame, group: str, order=None) -> pd.DataFrame:
    """Condo-fee summary per bucket, including the share within budget."""
    grouped = frame.groupby(group, observed=True)["condo_fee"]
    out = pd.DataFrame({
        "listings": grouped.size(),
        "p25": grouped.quantile(0.25).round(0),
        "median": grouped.median().round(0),
        "p75": grouped.quantile(0.75).round(0),
        "p95": grouped.quantile(0.95).round(0),
        f"% ≤ {brl(BUDGET)}": (grouped.apply(lambda s: (s <= BUDGET).mean()) * 100).round(1),
    })
    if order is not None:
        out = out.reindex([o for o in order if o in out.index])
    return out.rename_axis(group)


overall = pd.DataFrame({
    "listings": [len(df)],
    "p25": [round(df["condo_fee"].quantile(0.25))],
    "median": [round(df["condo_fee"].median())],
    "p75": [round(df["condo_fee"].quantile(0.75))],
    "p95": [round(df["condo_fee"].quantile(0.95))],
    f"% ≤ {brl(BUDGET)}": [round((df["condo_fee"] <= BUDGET).mean() * 100, 1)],
}, index=["All Aclimação apartments"])
overall

In [ ]:
for column, order in [
    ("bedroom_group", BEDROOM_ORDER),
    ("area_group", AREA_ORDER),
    ("parking_group", PARKING_ORDER),
    ("price_group", PRICE_ORDER),
    ("family_size", None),
]:
    print(f"\n### by {column}")
    display(summarise(df, column, order))

## 6. The histograms

### Plotting helpers

Shared so every chart is directly comparable: the same **R$ 250 bins**, the same
`percent` normalisation (each group sums to 100%, so a small group is not flattened by a
big one), and the same dashed R$ 2.000 reference line.

Two forms are used, and the choice is not cosmetic:

* **two or three groups** → overlaid histograms, one categorical colour each, legend
  always present and carrying the share within budget;
* **four groups** → small multiples in a single colour. Four overlapping translucent
  fills in one frame stop being readable, and past three slots this palette can't hold
  its colourblind-separation floor for every pair, so position does the work instead of
  hue.

In [ ]:
BIN_SIZE = 250
X_MAX = int(np.ceil(df["condo_fee"].max() / BIN_SIZE) * BIN_SIZE)
XBINS = dict(start=0, end=X_MAX, size=BIN_SIZE)

HOVER = (
    "Condo fee: R$ %{x}<br>"
    "%{y:.1f}% of the group"
    "<extra>%{fullData.name}</extra>"
)


def _style(fig, title, subtitle=None, show_legend=True, height=460):
    """Apply the shared chrome: recessive grid, hairline axes, quiet ink."""
    heading = f"<b>{title}</b>"
    if subtitle:
        heading += f"<br><sup>{subtitle}</sup>"
    fig.update_layout(
        title=dict(text=heading, x=0, xanchor="left", y=0.94, yanchor="top",
                   font=dict(size=17, color=T["ink"])),
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif',
                  size=12, color=T["ink_2"]),
        paper_bgcolor=T["surface"],
        plot_bgcolor=T["surface"],
        separators=",.",          # Brazilian number formatting on axes and hover
        bargap=0.08,              # surface gap between adjacent bars
        showlegend=show_legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
                    title_text="", font=dict(color=T["ink_2"])),
        margin=dict(l=70, r=40, t=130 if show_legend else 110, b=70),
        height=height,
        hoverlabel=dict(bgcolor=T["surface"], bordercolor=T["axis"],
                        font=dict(color=T["ink"], size=12)),
    )
    axis_common = dict(gridcolor=T["grid"], linecolor=T["axis"], zeroline=False,
                       showline=True, ticks="outside", tickcolor=T["axis"],
                       title_font=dict(color=T["muted"]),
                       tickfont=dict(color=T["muted"]))
    fig.update_xaxes(title_text="Monthly condo fee (R$)", tickprefix="R$ ",
                     tickformat=",.0f", range=[0, X_MAX], **axis_common)
    fig.update_yaxes(title_text="Share of listings (%)", ticksuffix="%", **axis_common)
    # The threshold the whole notebook is about. Dashed *because* it is a threshold --
    # the gridlines stay solid hairlines.
    fig.add_vline(x=BUDGET, line_width=2, line_dash="dash", line_color=T["ink"],
                  annotation_text=f"  {brl(BUDGET)} budget",
                  annotation_position="top right",
                  annotation_font=dict(color=T["ink"], size=12))
    return fig


def label_for(frame: pd.DataFrame, name: str) -> str:
    """Legend / facet label that carries the value, not just the identity."""
    share = (frame["condo_fee"] <= BUDGET).mean() * 100
    return f"{name} · {share:.0f}% ≤ {brl(BUDGET)} · n={len(frame):,}"


def hist_overlay(frame: pd.DataFrame, group: str, title: str, subtitle=None, order=None):
    """Two or three groups overlaid, one categorical slot each."""
    order = order or list(pd.unique(frame[group]))
    assert len(order) <= 3, "past three overlaid groups, use hist_facets instead"
    labels = {name: label_for(frame[frame[group] == name], name) for name in order}
    plotted = frame.assign(**{group: frame[group].map(labels)})
    fig = px.histogram(
        plotted, x="condo_fee", color=group, histnorm="percent", barmode="overlay",
        opacity=0.72, category_orders={group: [labels[n] for n in order]},
        color_discrete_sequence=T["series"][: len(order)],
    )
    fig.update_traces(
        xbins=XBINS, hovertemplate=HOVER,
        # 2px surface ring so overlapping fills stay separable
        marker_line=dict(width=1.5, color=T["surface"]),
    )
    return _style(fig, title, subtitle, show_legend=True)


def hist_facets(frame: pd.DataFrame, group: str, title: str, subtitle=None, order=None):
    """Four groups as small multiples: one colour, position carries identity."""
    
    order = order or list(pd.unique(frame[group]))
    labels = {name: label_for(frame[frame[group] == name], name) for name in order}
    plotted = frame.assign(**{group: frame[group].astype(str).map(labels)})
    
    fig = px.histogram(
        plotted, x="condo_fee", facet_col=group, facet_col_wrap=2,
        histnorm="percent", category_orders={group: [labels[n] for n in order]},
        color_discrete_sequence=[T["series"][0]],
    )
    
    fig.update_traces(xbins=XBINS, hovertemplate=HOVER,
                      marker_line=dict(width=1.5, color=T["surface"]))
    fig.update_yaxes(range=[0, None])
    fig.for_each_annotation(
        lambda a: a.update(text=a.text.split("=", 1)[-1],
                           font=dict(color=T["ink"], size=12))
    )
    
    fig = _style(fig, title, subtitle, show_legend=False, height=1200, width=1600)
    fig.update_layout(margin=dict(l=70, r=40, t=150, b=70))
    return fig

### 6.1 The whole neighbourhood

The baseline. The distribution is strongly right-skewed: the bulk of Aclimação
apartments sit far below R$ 2.000, and a long tail of large flats runs well past it.

In [ ]:
fig = px.histogram(df, x="condo_fee", histnorm="percent",
                   color_discrete_sequence=[T["series"][0]])
fig.update_traces(xbins=XBINS, name="All apartments", hovertemplate=HOVER,
                  marker_line=dict(width=1.5, color=T["surface"]))
share = (df["condo_fee"] <= BUDGET).mean() * 100
fig = _style(
    fig,
    "Condo fees, all apartments for sale in Aclimação/Vila Mariana",
    f"{share:.0f}% of {len(df):,} listings are at or below {brl(BUDGET)}/month "
    f"· median {brl(df['condo_fee'].median())}",
    show_legend=False,
)
fig.show()

### 6.2 By number of bedrooms

Four buckets, so small multiples. The mode marches right with every bedroom added, and
between 3 and 4 bedrooms the distribution crosses the budget line entirely.

In [ ]:
hist_facets(df, "bedroom_group",
            "Condo fees by number of bedrooms",
            f"Each panel is normalised to its own 100%. Dashed line = {brl(BUDGET)}/month.",
            order=BEDROOM_ORDER).show()

### 6.3 2+ bedrooms vs. one bedroom

The cut asked for directly. Requiring a second bedroom shifts the distribution right,
but not out of budget — the mass is still overwhelmingly under R$ 2.000.

In [ ]:
hist_overlay(df, "bedrooms_2plus",
             "Condo fees: 2+ bedrooms vs. one bedroom or studio",
             "Overlaid and each normalised to 100%, so group sizes don't distort the shapes.",
             order=["2+ bedrooms", "1 bedroom or studio"]).show()

### 6.4 By usable area

Area is the sharper predictor of the fee than bedroom count — unsurprising, since São
Paulo condo fees are usually apportioned by *fração ideal*, roughly the unit's share of
the building's floor area.

In [ ]:
hist_facets(df, "area_group",
            "Condo fees by usable area",
            f"Usable area in m². Dashed line = {brl(BUDGET)}/month.",
            order=AREA_ORDER).show()

### 6.5 100 m² and above vs. below

The second cut asked for. This is where R$ 2.000 stops being comfortable: the ≥ 100 m²
distribution straddles the line instead of sitting to the left of it.

In [ ]:
hist_overlay(df, "area_160plus",
             "Condo fees: apartments of 160 m² or more vs. smaller",
             "The 160 m² threshold splits the neighbourhood into two clearly different fee regimes.",
             order=["≥ 160 m²", "< 160 m²"]).show()

### 6.6 By parking spaces

A garage space is billed through the condo fee, so it doubles as a proxy for how much of
the building each unit owns.

In [ ]:
hist_overlay(df, "parking_group",
             "Condo fees by number of parking spaces",
             f"Three groups, three fixed colour slots. Dashed line = {brl(BUDGET)}/month.",
             order=PARKING_ORDER).show()

### 6.7 By asking price

Does a cheaper apartment come with a cheaper fee? Mostly yes — but the overlap between
the price bands is wide enough that price alone is a poor guide to the fee.

In [ ]:
hist_overlay(df, "price_group",
             "Condo fees by asking price band",
             "Wide overlap: the asking price is a weak predictor of the monthly fee.",
             order=PRICE_ORDER).show()

### 6.8 The family-size apartment

Combining both constraints — 3+ bedrooms **and** at least 100 m² — is the case where the
R$ 2.000 budget genuinely bites.

In [ ]:
hist_overlay(df, "family_size",
             "Condo fees: 3+ bedrooms and <= 160 m², vs. everything else",
             f"Both constraints together push most of the distribution past {brl(BUDGET)}.",
             order=["3+ bedrooms and <= 160 m²", "Everything else"]).show()

### 6.9 Reading the budget off a cumulative curve

The histograms show *shape*; this shows the answer to "what fraction fits?" at any
budget, not just R$ 2.000. Where each curve crosses the dashed line is that group's
share within budget.

In [ ]:
ecdf_groups = ["≤ 2 bedrooms", "3 bedrooms", "4+ bedrooms"]
ecdf_df = df.assign(
    ecdf_group=pd.cut(df["bedrooms"], bins=[-1, 2, 3, 99], labels=ecdf_groups, ordered=True)
)
labels = {g: label_for(ecdf_df[ecdf_df["ecdf_group"] == g], g) for g in ecdf_groups}
ecdf_df["ecdf_group"] = ecdf_df["ecdf_group"].astype(str).map(labels)

fig = px.ecdf(
    ecdf_df, x="condo_fee", color="ecdf_group", ecdfnorm="percent",
    category_orders={"ecdf_group": [labels[g] for g in ecdf_groups]},
    color_discrete_sequence=T["series"],
)
fig.update_traces(line=dict(width=2),
                  hovertemplate="Condo fee: R$ %{x}<br>%{y:.0f}% at or below"
                                "<extra>%{fullData.name}</extra>")
fig = _style(
    fig,
    "Share of listings at or below a given condo fee",
    "Read a budget on the x-axis, read the share of apartments that fit on the y-axis.",
    show_legend=True,
)
fig.update_yaxes(title_text="Cumulative share of listings (%)", range=[0, 100])
fig.show()

## 7. Takeaways

In [ ]:
def share(mask) -> str:
    subset = df.loc[mask]
    if subset.empty:
        return "no listings"
    return (f"{(subset['condo_fee'] <= BUDGET).mean():.0%} of {len(subset):,} "
            f"(median {brl(subset['condo_fee'].median())})")


lines = [
    f"Budget: {brl(BUDGET)}/month · clean sample: {len(df):,} listings "
    f"({len(raw) - len(df):,} garbage rows removed)",
    "",
    f"All apartments                     {share(df.index.notna())}",
    f"2+ bedrooms                        {share(df['bedrooms'] >= 2)}",
    f"3+ bedrooms                        {share(df['bedrooms'] >= 3)}",
    f"4+ bedrooms                        {share(df['bedrooms'] >= 4)}",
    f"100 m² or more                     {share(df['total_area_m2'] >= 100)}",
    f"150 m² or more                     {share(df['total_area_m2'] >= 150)}",
    f"2+ parking spaces                  {share(df['vacancies'] >= 2)}",
    f"3+ bedrooms and 100 m² or more     {share((df['bedrooms'] >= 3) & (df['total_area_m2'] >= 100))}",
    "",
    f"Typical fee per m²: {brl(df['fee_per_m2'].median(), 2)}/m² "
    f"(p25 {brl(df['fee_per_m2'].quantile(0.25), 2)}, "
    f"p75 {brl(df['fee_per_m2'].quantile(0.75), 2)})",
    f"At that median rate, {brl(BUDGET)} buys roughly "
    f"{BUDGET / df['fee_per_m2'].median():.0f} m² of apartment.",
]
print("\n".join(lines))

**What the charts say**

* R$ 2.000 is a **generous** budget for a small or mid-sized apartment in Aclimação and
  a **tight** one for a large flat. Most of the neighbourhood's for-sale stock sits well
  below the line.
* **Usable area, not bedroom count, drives the fee.** The area facets separate far more
  cleanly than the bedroom facets, and fee-per-m² is stable across the neighbourhood —
  which is what you'd expect from fees apportioned by *fração ideal*.
* Asking for **2+ bedrooms costs almost nothing** in budget terms. Asking for
  **≥ 100 m² costs a lot**, and asking for **both 3+ bedrooms and ≥ 100 m²** is where
  the budget starts excluding most of the market.
* Use the cumulative curve in 6.9 to re-answer this for any other budget without
  re-cutting the data.

**Caveats**

* `condo_fee` is scraped from an advertiser-entered field, so even after cleaning it is
  self-reported and not audited. The Tukey fence on fee-per-m² removes the impossible
  values, not the merely optimistic ones.
* Roughly 5% of raw listings disclose no fee at all (stored as `0`) and are excluded.
  If non-disclosure correlates with a high fee, the shares above are slightly optimistic.
* Duplicates are removed by URL only; the same flat re-listed under a new URL by another
  advertiser still counts twice.